# Task 8 – Bayesian Statistical Methods

This notebook gives a small practical demonstration of:
1. Naïve Bayes
2. Bayesian Poisson regression
3. Posterior uncertainty
4. Posterior predictive checking
5. Bayesian decision making

The Bayesian model uses a 5,000-row sample so that it can run more easily on a normal laptop.

## 1. Import packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import pymc as pm
import arviz as az

print("PyMC version:", pm.__version__)
print("ArviZ version:", az.__version__)

If PyMC or ArviZ is missing, install them once in the VS Code terminal:

```bash
pip install pymc arviz
```

## 2. Load and prepare the dataset

In [ ]:
df = pd.read_csv("../dataset/freMTPL2freq.csv")

df = df[df["Exposure"] > 0].copy()
df["LogDensity"] = np.log1p(df["Density"])
df["ClaimOccurred"] = (df["ClaimNb"] > 0).astype(int)

print("Dataset shape:", df.shape)
df.head()

## 3. Naïve Bayes demonstration

Naïve Bayes is used here as a binary classifier:
- 0 = no claim
- 1 = at least one claim

This is useful for evaluating Naïve Bayes, but it does not preserve the full claim-count information.

In [ ]:
nb_features = ["VehAge", "DrivAge", "BonusMalus", "LogDensity"]

X = df[nb_features]
y = df["ClaimOccurred"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

naive_bayes = GaussianNB()
naive_bayes.fit(X_train, y_train)

nb_pred = naive_bayes.predict(X_test)

print("Accuracy:", accuracy_score(y_test, nb_pred))
print()
print("Confusion matrix:")
print(confusion_matrix(y_test, nb_pred))
print()
print("Classification report:")
print(classification_report(y_test, nb_pred, digits=4, zero_division=0))

### Naïve Bayes interpretation

Naïve Bayes is simple and fast, but it converts `ClaimNb` into claim / no-claim classification. This loses information about multiple claims and does not naturally model exposure-adjusted claim frequency.

## 4. Prepare a sample for Bayesian Poisson regression

In [ ]:
bayes_df = df.sample(
    n=min(5000, len(df)),
    random_state=42
).copy()

bayes_features = ["VehAge", "DrivAge", "BonusMalus", "LogDensity"]

feature_means = bayes_df[bayes_features].mean()
feature_stds = bayes_df[bayes_features].std()

for col in bayes_features:
    bayes_df[col + "_z"] = (
        bayes_df[col] - feature_means[col]
    ) / feature_stds[col]

print("Bayesian sample shape:", bayes_df.shape)

bayes_df[
    ["ClaimNb", "Exposure", "VehAge_z", "DrivAge_z", "BonusMalus_z", "LogDensity_z"]
].head()

## 5. Bayesian Poisson regression

The response is `ClaimNb`. `log(Exposure)` is included as an offset, so the model estimates claim frequency while accounting for different exposure periods.

In [ ]:
y_claims = bayes_df["ClaimNb"].to_numpy()
exposure = bayes_df["Exposure"].to_numpy()

vehage_z = bayes_df["VehAge_z"].to_numpy()
drivage_z = bayes_df["DrivAge_z"].to_numpy()
bonus_z = bayes_df["BonusMalus_z"].to_numpy()
density_z = bayes_df["LogDensity_z"].to_numpy()

with pm.Model() as bayes_model:

    intercept = pm.Normal("intercept", mu=-2.3, sigma=1.0)

    b_vehage = pm.Normal("b_vehage", mu=0, sigma=0.5)
    b_drivage = pm.Normal("b_drivage", mu=0, sigma=0.5)
    b_bonus = pm.Normal("b_bonus", mu=0, sigma=0.5)
    b_density = pm.Normal("b_density", mu=0, sigma=0.5)

    eta = (
        intercept
        + b_vehage * vehage_z
        + b_drivage * drivage_z
        + b_bonus * bonus_z
        + b_density * density_z
        + np.log(exposure)
    )

    mu = pm.math.exp(eta)

    claims = pm.Poisson(
        "claims",
        mu=mu,
        observed=y_claims
    )

    trace = pm.sample(
        draws=500,
        tune=500,
        chains=2,
        cores=1,
        target_accept=0.90,
        random_seed=42,
        progressbar=True
    )

## 6. Posterior summary

In [ ]:
summary = az.summary(
    trace,
    var_names=[
        "intercept",
        "b_vehage",
        "b_drivage",
        "b_bonus",
        "b_density"
    ]
)

summary

The posterior mean shows the estimated direction and size of each coefficient. `r_hat` values close to 1 indicate good agreement between the sampling chains.

## 7. Manual 95% posterior credible intervals

In [ ]:
parameters = [
    "b_vehage",
    "b_drivage",
    "b_bonus",
    "b_density"
]

results = []

for param in parameters:
    samples = trace.posterior[param].values.flatten()

    results.append({
        "Parameter": param,
        "Mean": np.mean(samples),
        "Lower_95": np.quantile(samples, 0.025),
        "Upper_95": np.quantile(samples, 0.975)
    })

posterior_results = pd.DataFrame(results)

posterior_results

This calculates the 95% posterior interval directly from the posterior samples, so it does not depend on ArviZ HDI column names.

## 8. Plot the Bayesian coefficients

In [ ]:
means = posterior_results["Mean"].to_numpy()
lower = posterior_results["Lower_95"].to_numpy()
upper = posterior_results["Upper_95"].to_numpy()

y_positions = np.arange(len(posterior_results))

lower_error = means - lower
upper_error = upper - means

plt.figure(figsize=(8, 4))

plt.errorbar(
    means,
    y_positions,
    xerr=[lower_error, upper_error],
    fmt="o",
    capsize=4
)

plt.axvline(x=0, linestyle="--")

plt.yticks(
    y_positions,
    posterior_results["Parameter"]
)

plt.xlabel("Posterior coefficient")
plt.ylabel("Parameter")
plt.title("Bayesian Poisson Regression Coefficients")

plt.tight_layout()
plt.show()

### Plot interpretation

- A positive coefficient suggests higher expected claim frequency.
- A negative coefficient suggests lower expected claim frequency.
- If the full 95% posterior interval is above or below zero, there is stronger posterior evidence about the direction of the effect.

## 9. Posterior predictive check

In [ ]:
with bayes_model:
    posterior_predictive = pm.sample_posterior_predictive(
        trace,
        random_seed=42,
        progressbar=True
    )

predicted_claims = (
    posterior_predictive
    .posterior_predictive["claims"]
    .values
)

observed_mean = y_claims.mean()
predicted_mean = predicted_claims.mean()

print("Observed mean claim count:", observed_mean)
print("Posterior predictive mean claim count:", predicted_mean)
print("Difference:", predicted_mean - observed_mean)

The posterior predictive mean should be reasonably close to the observed mean if the model is reproducing the general claim-count level of the sample.

## 10. Bayesian decision-making example

In [ ]:
portfolio_rate = (
    bayes_df["ClaimNb"].sum()
    / bayes_df["Exposure"].sum()
)

policy = bayes_df.iloc[0]

posterior = trace.posterior.stack(
    sample=("chain", "draw")
)

intercept_samples = posterior["intercept"].values
vehage_samples = posterior["b_vehage"].values
drivage_samples = posterior["b_drivage"].values
bonus_samples = posterior["b_bonus"].values
density_samples = posterior["b_density"].values

log_rate_samples = (
    intercept_samples
    + vehage_samples * policy["VehAge_z"]
    + drivage_samples * policy["DrivAge_z"]
    + bonus_samples * policy["BonusMalus_z"]
    + density_samples * policy["LogDensity_z"]
)

rate_samples = np.exp(log_rate_samples)

posterior_mean_rate = rate_samples.mean()

prob_above_portfolio = np.mean(
    rate_samples > portfolio_rate
)

print("Portfolio claim frequency:", portfolio_rate)
print("Policy posterior mean claim frequency:", posterior_mean_rate)
print(
    "Probability policy risk is above portfolio average:",
    prob_above_portfolio
)

## 11. Example decision rule

In [ ]:
decision_threshold = 0.80

if prob_above_portfolio > decision_threshold:
    decision = "Flag for additional underwriting review"
else:
    decision = "No additional underwriting review flag"

print("Decision:", decision)

The 80% threshold is only an illustrative decision rule. In a real insurer, the threshold should be selected using business costs, risk appetite, regulation and expert judgement.

## 12. Final Task 8 interpretation

The practical demonstrations support the critical evaluation in Task 8.

**Naïve Bayes:** simple and computationally efficient, but less suitable as the main model because it changes the original claim-count problem into a binary classification problem.

**Bayesian Poisson regression:** more directly suited to the current project because it models claim counts, accounts for exposure and provides posterior distributions for model parameters.

**Bayesian decision making:** posterior probabilities can be converted into decision-support rules, such as identifying policies that may require additional underwriting review.

These methods should be considered complementary to the frequentist models developed in Task 5 rather than automatically superior to them.